# 1019 32일차

## 1. LangGraph

체인의 단계를 노드로 두고, 노드 사이 연결을 직접 정해서 실행 흐름을 만드는 라이브러리

LCEL 체인(질문 → 검색 → 모델 → 답)은 Sequential처럼 한 방향으로만 이어짐. 조건에 따라 다른 단계로 보내려면(분기) 함수형 모델처럼 연결을 직접 정해야 함

1. State : 노드들이 같이 읽고 쓰는 값. `TypedDict`로 키와 타입을 정함
2. Node : State를 받아 바꿀 키만 dict로 돌려주는 함수
3. Edge : 노드 사이 연결. `START`에서 시작해 `END`에서 끝남

## 2. TypedDict

dict의 키마다 값 타입을 적어 두는 클래스 (`lg01_TypedDict`)

State에 어떤 키가 있고 각 값이 어떤 타입인지 정하는 데 씀

1. `Dict[str, str]` : 모든 키와 값의 타입을 한 번에 지정
2. `class 이름(TypedDict)` : 키마다 타입을 따로 지정. 괄호 안 `TypedDict`는 상속

- 만드는 방법은 `Person(name=..., age=...)` 또는 `p: Person = {...}`, 결과는 일반 dict

#### cf) 타입 강제 없음

- 정한 타입과 다른 값을 넣어도 실행 에러가 나지 않음
- 타입이 다르다는 표시는 Pylance 같은 편집기 검사에서만 나옴

## 3. add_messages

두 메시지 리스트를 합치는 리듀서 (`lg02_add_messages`)

노드가 돌려준 메시지로 `messages`를 덮어쓰지 않고, 대화 기록 뒤에 붙이려고 씀

1. 리듀서 : (기존 값, 새 값)을 받아 합친 값을 돌려주는 함수
2. `Annotated[list, add_messages]` : State의 키에 리듀서를 지정

- id가 다르면 기존 리스트 뒤에 붙임
- id가 같으면 기존 메시지를 새 메시지로 바꿈
- `("user", "문자열")` 튜플은 `HumanMessage`로 바뀜
- 리듀서가 없는 키는 노드가 돌려준 값으로 덮어씀

## 4. 그래프 만들기

State → Node → Edge 순서로 정의하고 컴파일해서 실행함 (`lg03_chatbot`)

1. State 정의 : `class State(TypedDict)`
2. Node 정의 : `def chatbot(state)`가 `{"messages": [llm.invoke(state["messages"])]}`를 돌려줌
3. Graph 생성 : `StateGraph(State)`에 `add_node("이름", 함수)`
4. Edge 추가 : `add_edge(START, "이름")`, `add_edge("이름", END)`
5. 컴파일 : `compile()`로 실행할 수 있는 그래프를 만듦
6. 실행 : `invoke({"messages": [...]})`

- 결과는 State dict. `response["messages"]`에 질문(`HumanMessage`)과 답(`AIMessage`)이 차례로 들어 있음
- `.type`은 `human`·`ai`, 답 문자열은 `.content`
- 값을 넣은 적이 없는 키는 결과에 없음

## 5. 노드 연결

노드를 여러 개 만들고 `add_edge`로 차례로 이어 실행 순서를 정하는 방법 (`lg04_add_node_edge`)

1. `add_node` : 노드마다 이름과 함수를 등록
2. `add_edge(앞 노드, 뒤 노드)` : `START → 노드1 → 노드2 → … → END` 순서로 하나씩 연결

- 앞 노드가 바꾼 State가 다음 노드에 그대로 넘어감
- 노드마다 돌려준 메시지가 `add_messages`로 실행 순서대로 `messages`에 쌓임
- 모델을 부르는 노드에는 앞 노드가 붙인 메시지까지 대화 기록으로 들어감